# Codebase Search with ripgrep and tree-sitter

In this demo, we will compare what ripgrep and tree-sitter return for the same search over [python-accounting](https://github.com/ekmungai/python-accounting) codebase.

In [ ]:
from pathlib import Path

ROOT = Path("python-accounting").resolve()
PACKAGE = ROOT / "python_accounting"
# Look for code that posts a transaction to the ledger
SEARCH_TERM = "post"


def relative(path: str | Path) -> str:
    return str(Path(path).relative_to(ROOT))

## 1. Search with ripgrep

Search for the term as text across the package.

In [ ]:
import re

from ripgrep_rs import search_structured

matches = search_structured(
    patterns=[SEARCH_TERM], paths=[str(PACKAGE)], globs=["*.py"]
)
print(f"ripgrep found {len(matches)} lines containing '{SEARCH_TERM}'. The first 15:\n")
for match in matches[:15]:
    print(
        f"  {relative(match.path)}:{match.line_number}: {match.line_text.strip()[:70]}"
    )

definition_line = rf"\s*(def|class)\s+\w*{SEARCH_TERM}"
definitions = [m for m in matches if re.match(definition_line, m.line_text)]
print(
    f"\nOnly {len(definitions)} of those lines define a class or function with '{SEARCH_TERM}' in its name."
)

## 2. How tree-sitter represents a module

Parse one module and print the structure of its syntax tree, with line numbers.

In [20]:
import tree_sitter_python as tspython
from tree_sitter import Language, Node, Parser

PY_LANGUAGE = Language(tspython.language())
PARSER = Parser(PY_LANGUAGE)

MODULE = PACKAGE / "models" / "ledger.py"

TREE_NODES = {
    "class_definition",
    "block",
    "decorated_definition",
    "function_definition",
}


def print_tree(node: Node, depth: int = 0, max_depth: int = 5) -> None:
    """Print a node and its named children, with line numbers from 1."""
    name = node.child_by_field_name("name")
    label = f" '{name.text.decode()}'" if name else ""
    print(
        f"{'  ' * depth}{node.type}{label} [{node.start_point[0] + 1}-{node.end_point[0] + 1}]"
    )
    if depth < max_depth:
        for child in node.named_children:
            if child.type in TREE_NODES:
                print_tree(child, depth + 1, max_depth)


tree = PARSER.parse(MODULE.read_bytes())
print(f"Syntax tree of {relative(MODULE)}:\n")
print_tree(tree.root_node)

Syntax tree of python_accounting/models/ledger.py:

module [1-272]
  class_definition 'Ledger' [25-271]
    block [28-271]
      function_definition '__repr__' [79-83]
        block [80-83]
      decorated_definition [85-119]
        function_definition '_allocate_amount' [86-119]
          block [89-119]
      decorated_definition [121-134]
        function_definition '_make_compound_ledgers' [122-134]
          block [129-134]
      decorated_definition [136-156]
        function_definition '_post_compound' [137-156]
          block [139-156]
      decorated_definition [158-173]
        function_definition '_transaction_ledgers' [159-173]
          block [160-173]
      decorated_definition [175-219]
        function_definition '_post_simple' [176-219]
          block [177-219]
      decorated_definition [221-233]
        function_definition 'post' [222-233]
          block [223-233]
      function_definition 'get_hash' [235-271]
        block [236-271]


## 3. Search with tree-sitter

A query describes one or more patterns to look for in the tree.

Each pattern is an [S-expression](https://en.wikipedia.org/wiki/S-expression) that matches a certain set of nodes in a syntax tree. The expression to match a given node consists of a pair of parentheses containing two things: 
- The node’s type
- and optionally, a series of other S-expressions that match the node’s children.

For example, in the query below, we are looking for class and function definitions whose name contains the same search term.

Read more about [queries](https://tree-sitter.github.io/tree-sitter/using-parsers/queries/1-syntax.html), [captures](https://tree-sitter.github.io/tree-sitter/using-parsers/queries/2-operators.html#capturing-nodes) and [predicates](https://tree-sitter.github.io/tree-sitter/using-parsers/queries/3-predicates-and-directives.html#the-match-predicate)

In [ ]:
from tree_sitter import Query, QueryCursor

SEARCH_DEFINITIONS = Query(
    PY_LANGUAGE,
    f"""
    (class_definition name: (identifier) @name
      (#match? @name "{SEARCH_TERM}")) @class
    (function_definition name: (identifier) @name
      (#match? @name "{SEARCH_TERM}")) @function
    """,
)

In [17]:
def definitions_in(path: Path, query: Query) -> list[dict]:
    """The classes and functions a query finds in a file, with line spans."""
    tree = PARSER.parse(path.read_bytes())
    symbols = []
    for pattern, captures in QueryCursor(query).matches(tree.root_node):
        kind = "class" if pattern == 0 else "function"
        node = captures[kind][0]
        symbols.append(
            {
                "name": captures["name"][0].text.decode(),
                "kind": kind,
                "top_level": node.start_point[1] == 0,  # methods are indented
                "path": relative(path),
                "start_line": node.start_point[0] + 1,
                "end_line": node.end_point[0] + 1,
            }
        )
    return sorted(symbols, key=lambda s: s["start_line"])

In [ ]:
matches = [
    s
    for path in sorted(PACKAGE.rglob("*.py"))
    for s in definitions_in(path, SEARCH_DEFINITIONS)
]
print(
    f"tree-sitter found {len(matches)} definitions with '{SEARCH_TERM}' in their name:\n"
)
for s in matches:
    print(
        f"  {s['kind']:<8} {s['name']:<24} {s['path']}:{s['start_line']}-{s['end_line']}"
    )

tree-sitter found 7 definitions with 'post' in their name:

  class    UnpostedAssignmentError  python_accounting/exceptions/__init__.py:338-343
  function is_posted                python_accounting/models/balance.py:89-91
  function _post_compound           python_accounting/models/ledger.py:137-156
  function _post_simple             python_accounting/models/ledger.py:176-219
  function post                     python_accounting/models/ledger.py:222-233
  function is_posted                python_accounting/models/transaction.py:184-186
  function post                     python_accounting/models/transaction.py:231-253


## 4. Outline the whole package

Remove the condition on the name, and the same query finds every definition. Use that to build an index, then print a compact outline without the methods.

In [15]:
DEFINITIONS = Query(
    PY_LANGUAGE,
    """
    (class_definition name: (identifier) @name) @class
    (function_definition name: (identifier) @name) @function
    """,
)

In [18]:
index = [
    s
    for path in sorted(PACKAGE.rglob("*.py"))
    for s in definitions_in(path, DEFINITIONS)
]
lines = []
for path in sorted({s["path"] for s in index}):
    names = [s["name"] for s in index if s["path"] == path and s["top_level"]]
    lines.append(f"{path}: {', '.join(names)}")
outline = "\n".join(lines)
print(outline)

source_chars = sum(len(p.read_text(encoding="utf-8")) for p in PACKAGE.rglob("*.py"))
print(f"\nSymbols indexed: {len(index)}")
print(
    f"Source code:     {source_chars:>7} characters, about {source_chars // 4} tokens"
)
print(
    f"This outline:    {len(outline):>7} characters, about {len(outline) // 4} tokens"
)

python_accounting/config.py: Config
python_accounting/database/accounting_functions.py: AccountingFunctionsMixin
python_accounting/database/database_init.py: database_init
python_accounting/database/event_listeners.py: _filter_options, EventListenersMixin
python_accounting/database/session.py: AccountingSession, get_session
python_accounting/database/session_overrides.py: SessionOverridesMixin
python_accounting/exceptions/__init__.py: AccountingExeption, AdjustingReportingPeriodError, ClosedReportingPeriodError, DuplicateReportingPeriodError, InvalidAccountTypeError, HangingTransactionsError, InvalidBalanceAccountError, InvalidBalanceDateError, InvalidBalanceTransactionError, InvalidCategoryAccountTypeError, InvalidLineItemAccountError, InvalidMainAccountError, InvalidTaxAccountError, InvalidTaxChargeError, InvalidTransactionDateError, InvalidTransactionTypeError, MissingEntityError, MissingLineItemError, MissingMainAccountAmountError, MissingReportingPeriodError, MissingTaxAccountErro